# Homes on Dangerous Streets?
> **Owner:** Mohamed

## Team

| Member | ID |
|---|---|
| Mohamed nazmy | |
| Mostafa Ayman | |
| Mostafa Alaa | |
| Ali | |
| Moustafa Maged | |
| baraa | |

## Contributions

| Member | Contributions |
|---|---|
| Mohamed nazmy | |
| Mostafa Ayman | |
| Mostafa Alaa | |
| Ali | |
| Moustafa Maged | |
| baraa | |

## 1. Introduction
> **Owner:** Mostafa Ayman

### Main question

### Big idea

### Why it matters to New Yorkers

## 2. Datasets and why we chose them
> **Owner:** Mohamed

| | Housing & Development | Transportation |
|---|---|---|
| Dataset | [Affordable Housing Production by Building](https://data.cityofnewyork.us/Housing-Development/Affordable-Housing-Production-by-Building/hg8x-zxpr/about_data) (catalog H2) | [Serious Injury Response Tracking & Analysis Program (SIRTA)](https://data.cityofnewyork.us/Transportation/Serious-Injury-Response-Tracking-Analysis-Program-/xeqp-qz8h/about_data) (catalog T4) |
| Dataset ID | `hg8x-zxpr` | `xeqp-qz8h` |
| Size | 9,551 rows × 41 columns | 13,754 rows × 23 columns |
| One row = | one affordable building | one crash with a severe injury or death |
| Geographic fields | Borough, Postcode (ZIP), Latitude/Longitude | Borough, ZipCode, Latitude/Longitude, street |
| Time fields | Project Start Date, Building Completion Date | Crash Date, Crash Time |

**Why these two.** Our question is whether New York is building its affordable homes on dangerous streets. New affordable buildings add residents who walk, bike and take transit, so the safety of the streets around them matters. The housing dataset tells us where and when affordable buildings were completed and how big they are; SIRTA tells us where and when serious crashes happened. Both are in the course catalog, and both carry the same three geographic fields (borough, ZIP code, coordinates) plus dates, so they can be linked by place and time.

**Size.** Both datasets are small enough to download in full with the loading code from the project description, so no sample was needed.

**Analysis window.** We use 1 Jan 2022 – 30 Jun 2026, the only period both datasets cover. Housing runs from 2014; SIRTA starts in 2022.

**How the datasets will be joined.** A building and a crash never share a record, and the datasets have different row types (buildings vs crashes), so we do not join row to row. Instead we bring both to a common level:

| Level | Used for | Why this level |
|---|---|---|
| ZIP code (and ZIP × year, ZIP × month) | Most questions | Present in both datasets and much finer than borough |
| Borough × year | Q12 | Borough is complete in both datasets, so every record counts |
| Building-level distance (crashes within 100 m / 250 m of a building) | Q8, Q11 | "Doorstep" questions need exact locations, not areas |

**Keys we did not use.** Neighborhood Tabulation Area and census tract use 2010 codes in the housing data but 2020 codes in SIRTA, so they don't match. Council district and community board are mostly empty in SIRTA.

**Known gaps that affect the join** (details in sections 4–7):
- 1,958 housing buildings (20.5%) have no ZIP or coordinates; 1,949 of them are confidential projects. They can be used at borough level only.
- 41.3% of SIRTA crashes have no ZIP code. We plan to fill ZIPs from crash coordinates where possible (section 8).


## 3. Setup
> **Owner:** Mohamed

Imports and constants.

In [1]:
import pandas as pd

# Replace with the dataset IDs found in each dataset's URL
housing_id = 'hg8x-zxpr'
transport_id = 'xeqp-qz8h'

base_url = 'https://data.cityofnewyork.us/api/views/{}/rows.csv?accessType=download'

# Load the chosen datasets
df_housing = pd.read_csv(base_url.format(housing_id), low_memory=False)
df_transport = pd.read_csv(base_url.format(transport_id), low_memory=False)

# Quick preview

df_transport.head()


,Collision ID,Crash Date,Crash Time,Crash Police Precinct,Crash Borough,Total Non-Severe Injuries,Total Severe Injuries,Total Fatalities,Latitude,Longitude,...,Highway Reference Marker,ZipCode,Similar Fatalities & Severe Injuries (3Y Lookback),Similar Fatalities & All Injuries (3Y Lookback),Community Board,Council District,Building Identification Number (BIN),Borough Block Lot (BBL),Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
0,4492364.0,01/01/2022,01:28,46.0,Bronx,1.0,1.0,0.0,40.858707,-73.90370,...,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,4491159.0,01/01/2022,04:33,46.0,Bronx,0.0,1.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,4491487.0,01/01/2022,05:55,47.0,Bronx,0.0,1.0,0.0,40.871550,-73.86428,...,NaN,NaN,0.0,0.0,212.0,12.0,2056281.0,2.045980e+09,372.0,BX1201
3,4491223.0,01/01/2022,10:42,43.0,Bronx,0.0,1.0,0.0,40.823673,-73.87233,...,907HX1M11001,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,4491418.0,01/01/2022,20:55,49.0,Bronx,0.0,1.0,0.0,40.863945,-73.86452,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import sys
sys.path.append("../src")   # lets the notebook import files from src/
import charts

charts.CHARTS["q1"]({}, charts.DEFAULT_FILTERS).show()


## 4. Housing: overview and EDA
> **Owner:** Mohamed

In [3]:
df_housing.head()


,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,2-BR Units,3-BR Units,4-BR Units,5-BR Units,6-BR+ Units,Unknown-BR Units,Counted Rental Units,Counted Homeownership Units,All Counted Units,Total Units
0,68872,177-06 WEXFORD TERRACE,06/30/2026,NaN,991637.0,177-06,WEXFORD TERRACE,Queens,11432.0,4.098350e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,9.0,43
1,72436,ST. NICK'S. 819 GRAND STREET,06/30/2026,NaN,1021347.0,819,GRAND STREET,Brooklyn,11211.0,3.029220e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,187.0,NaN,187.0,187
2,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,21199.0,127,EDGECOMBE AVENUE,Manhattan,10030.0,1.020480e+09,...,20.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,25.0,25
3,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,28145.0,602,ST NICHOLAS AVENUE,Manhattan,10030.0,1.020480e+09,...,5.0,NaN,NaN,NaN,NaN,NaN,16.0,NaN,16.0,16
4,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,40555.0,129,WEST 133 STREET,Manhattan,10030.0,1.019180e+09,...,6.0,4.0,NaN,NaN,NaN,NaN,15.0,NaN,15.0,15


In [4]:
print(df_housing.shape, df_transport.shape)
print(df_transport["ZipCode"].isna().mean())


(9551, 41) (13754, 23)
0.413116184382725


In [5]:
pd.set_option("display.max_columns", None)
print(df_housing.columns.tolist())


['Project ID', 'Project Name', 'Project Start Date', 'Project Completion Date', 'Building ID', 'Number', 'Street', 'Borough', 'Postcode', 'BBL', 'BIN', 'Community Board', 'Council District', 'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude', 'Longitude', 'Latitude (Internal)', 'Longitude (Internal)', 'Building Completion Date', 'Reporting Construction Type', 'Extended Affordability Only', 'Prevailing Wage Status', 'Extremely Low Income Units', 'Very Low Income Units', 'Low Income Units', 'Moderate Income Units', 'Middle Income Units', 'Other Income Units', 'Studio Units', '1-BR Units', '2-BR Units', '3-BR Units', '4-BR Units', '5-BR Units', '6-BR+ Units', 'Unknown-BR Units', 'Counted Rental Units', 'Counted Homeownership Units', 'All Counted Units', 'Total Units']


In [6]:
missing = pd.DataFrame({
    "dtype": df_housing.dtypes,
    "missing": df_housing.isna().sum(),
    "missing_%": (df_housing.isna().mean() * 100).round(1),
})
missing.sort_values("missing_%", ascending=False)


,dtype,missing,missing_%
6-BR+ Units,float64,9524,99.7
5-BR Units,float64,9478,99.2
4-BR Units,float64,8990,94.1
Unknown-BR Units,float64,8862,92.8
Other Income Units,float64,8191,85.8
Moderate Income Units,float64,7857,82.3
Extremely Low Income Units,float64,7470,78.2
Middle Income Units,float64,6860,71.8
Counted Homeownership Units,float64,6821,71.4
Studio Units,float64,6353,66.5


In [7]:
# Check 1: are the 1,958 rows without a ZIP the same rows without coordinates?
no_postcode = df_housing["Postcode"].isna()
print("No postcode AND no latitude:", (no_postcode & df_housing["Latitude"].isna()).sum())
df_housing.loc[no_postcode, ["Project Name", "Number", "Street", "Borough"]].head(10)


No postcode AND no latitude: 1958


,Project Name,Number,Street,Borough
101,CONFIDENTIAL,----,----,Brooklyn
102,CONFIDENTIAL,----,----,Bronx
103,CONFIDENTIAL,----,----,Bronx
104,CONFIDENTIAL,----,----,Brooklyn
105,CONFIDENTIAL,----,----,Brooklyn
106,CONFIDENTIAL,----,----,Brooklyn
130,CONFIDENTIAL,----,----,Staten Island
131,CONFIDENTIAL,----,----,Bronx
132,CONFIDENTIAL,----,----,Queens
133,CONFIDENTIAL,----,----,Brooklyn


In [8]:
# Check 2: does "empty" mean 0? If the income tiers (empty = 0) add up to
# All Counted Units, empty cells are zeros, not unknown values.
income_cols = ["Extremely Low Income Units", "Very Low Income Units", "Low Income Units",
               "Moderate Income Units", "Middle Income Units", "Other Income Units"]
income_sum = df_housing[income_cols].fillna(0).sum(axis=1)
print("Rows where tiers add up:", (income_sum == df_housing["All Counted Units"]).sum(), "of", len(df_housing))


Rows where tiers add up: 9522 of 9551


In [9]:
# Are all 1,958 location-less rows confidential?
print("CONFIDENTIAL rows:", (df_housing["Project Name"] == "CONFIDENTIAL").sum())

# Categories: how buildings split by borough and construction type
print(df_housing["Borough"].value_counts(), "\n")
print(df_housing["Reporting Construction Type"].value_counts(), "\n")

# Date range of building completions
completion = pd.to_datetime(df_housing["Building Completion Date"], format="%m/%d/%Y")
print("Completions from", completion.min(), "to", completion.max())
print(completion.dt.year.value_counts().sort_index())


CONFIDENTIAL rows: 1949
Borough
Brooklyn         3649
Bronx            2630
Manhattan        1485
Queens           1342
Staten Island     445
Name: count, dtype: int64 

Reporting Construction Type
Preservation        4785
New Construction    4766
Name: count, dtype: int64 

Completions from 2014-01-03 00:00:00 to 2026-08-21 00:00:00
Building Completion Date
2014.0    376
2015.0    290
2016.0    432
2017.0    830
2018.0    664
2019.0    862
2020.0    419
2021.0    581
2022.0    654
2023.0    806
2024.0    705
2025.0    792
2026.0    474
Name: count, dtype: int64


In [10]:
import plotly.express as px

# Summary statistics for the two unit columns
print(df_housing[["All Counted Units", "Total Units"]].describe())

# Box plot on a log scale: the data is strongly skewed, so a normal scale squashes the boxes
px.box(df_housing, x="Borough", y="All Counted Units", log_y=True,
       title="Affordable units per building, by borough (log scale)").show()

# Completions per year (2026 is a partial year, to 21 Aug)
years = completion.dt.year.value_counts().sort_index()
px.bar(x=years.index, y=years.values,
       labels={"x": "Year of building completion", "y": "Buildings completed"},
       title="Affordable buildings completed per year").show()


       All Counted Units  Total Units
count        9522.000000  9551.000000
mean           33.705734    46.653544
std            65.834105    92.482144
min             1.000000     1.000000
25%             2.000000     3.000000
50%             8.000000    12.000000
75%            30.000000    45.000000
max           917.000000  1320.000000


In [11]:
units = df_housing["All Counted Units"]
print("Above IQR fence (72):", (units > 72).sum())

size = df_housing["Total Units"]
print("Small (<50):", (size < 50).sum(),
      "| Medium (50-199):", size.between(50, 199).sum(),
      "| Large (200+):", (size >= 200).sum())


Above IQR fence (72): 1249
Small (<50): 7309 | Medium (50-199): 1698 | Large (200+): 544


In [12]:
# Duplicates: fully identical rows, and repeated building IDs within a project
print("Fully duplicated rows:", df_housing.duplicated().sum())
known_id = df_housing["Building ID"].notna()
print("Repeated Project ID + Building ID:",
      df_housing[known_id].duplicated(subset=["Project ID", "Building ID"]).sum())

# Consistency: affordable units should never exceed total units
print("Affordable > total:", (df_housing["All Counted Units"] > df_housing["Total Units"]).sum())

# Consistency: completion should not come before the project started
start = pd.to_datetime(df_housing["Project Start Date"], format="%m/%d/%Y")
print("Completed before start:", (completion < start).sum())

# Consistency: coordinates should fall inside NYC (roughly lat 40.4–41.0, lon -74.3 to -73.6)
lat, lon = df_housing["Latitude"], df_housing["Longitude"]
print("Outside NYC box:", (~lat.between(40.4, 41.0) & lat.notna()).sum() +
      (~lon.between(-74.3, -73.6) & lon.notna()).sum())


Fully duplicated rows: 0
Repeated Project ID + Building ID: 0
Affordable > total: 0
Completed before start: 20
Outside NYC box: 0


In [13]:
bad_dates = completion < start
check = df_housing.loc[bad_dates, ["Project ID", "Project Name", "Reporting Construction Type",
                                   "Project Start Date", "Building Completion Date"]].copy()
check["days_early"] = (start - completion)[bad_dates].dt.days
check.sort_values("days_early", ascending=False)


,Project ID,Project Name,Reporting Construction Type,Project Start Date,Building Completion Date,days_early
8132,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,01/08/2015,357
8129,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,02/07/2015,327
8127,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,02/26/2015,308
8126,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,04/02/2015,273
8125,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,05/04/2015,241
8135,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,11/30/2015,31
4714,70377,CONFIDENTIAL,New Construction,10/29/2020,10/09/2020,20
1340,76576,CONFIDENTIAL,New Construction,02/27/2025,02/21/2025,6
6423,61837,9 FORT WASHINGTON AVENUE CLUSTER,Preservation,06/25/2018,06/19/2018,6
6422,61837,9 FORT WASHINGTON AVENUE CLUSTER,Preservation,06/25/2018,06/19/2018,6


### 4.x Summary: issues found and what cleaning should do

| # | Issue found | Evidence | Suggested handling | Affects |
|---|---|---|---|---|
| 1 | Confidential buildings have no location | 1,949 rows named `CONFIDENTIAL`; 1,958 rows have no Postcode and no coordinates (same rows) | Keep. Use at borough level only; they drop out of ZIP and distance questions. Check the 9 non-confidential rows without location | Q1–Q11 (ZIP/distance) |
| 2 | Hidden missing values | `Number` and `Street` show 0 missing but use `----` as a placeholder in confidential rows | Replace `----` with missing | Address fields |
| 3 | Empty unit-breakdown cells | Income tiers (empty → 0) add up to `All Counted Units` in every row with a known total | Fill income tiers with 0. Run the same check on bedroom and tenure columns before filling | Q9 |
| 4 | Missing affordable unit count | 29 rows (0.3%) have no `All Counted Units` | Decide: exclude from unit totals, or fill from the tier sum if it is non-zero | Q1, Q7, Q12 |
| 5 | Missing completion date | 1,666 rows (17.4%) have no `Building Completion Date` | Keep; check whether these are unfinished buildings. Exclude from completion-based counts | Q1, Q2, Q7, Q12 |
| 6 | Dates stored as text | All 3 date columns are `str` in `MM/DD/YYYY`; all non-empty completion dates parse with this format | Convert with `pd.to_datetime(..., format="%m/%d/%Y")` | All time questions |
| 7 | ZIP stored as a decimal | `Postcode` is `float64` (`11432.0`) | Convert to a 5-character text ZIP (`"11432"`) | Join with SIRTA |
| 8 | IDs stored as decimals | `Building ID`, `BBL`, `BIN` are `float64` because of missing values | Convert to whole-number or text IDs | Duplicate checks, joins |
| 9 | Very large buildings | 1,249 buildings (13.1%) are above the IQR fence of 72 affordable units; the largest has 917 | Keep: they are real buildings, and Q2/Q6 study large buildings directly. Use domain rules (impossible values) instead of IQR; none found (minimum is 1 unit) | Q2, Q6, all unit totals |
| 10 | Completed before project start | 20 buildings in 5 projects; 14 of them 1–20 days early. Only one is in our window, and it is confidential | Keep and flag; no effect on our questions | Q2, Q6 (none in practice) |
| 11 | Completions after the window | 2026 completions run to 21 Aug 2026; SIRTA ends 30 Jun 2026 | Cut at 30 Jun 2026 | All questions |
| 12 | Two coordinate pairs | `Latitude`/`Longitude` missing in 1,958 rows; the Internal pair in 2,044 | Use `Latitude`/`Longitude`; drop the Internal pair | Q8, Q11 |
| 13 | Columns no question uses | Extended Affordability Only, Prevailing Wage Status, Census Tract, NTA, Community Board, Council District, Internal coordinates | Candidates to drop, to keep the table small | — |

**Checked and fine:** no duplicated rows; no repeated Project ID + Building ID; affordable units never exceed total units; all coordinates fall inside NYC; Borough has exactly 5 clean values; Reporting Construction Type has exactly 2 values (Preservation 4,785, New Construction 4,766).


## 5. SIRTA: overview and EDA
> **Owner:** Ali

One row = one crash in which at least one person was severely injured or killed. This section only looks at the raw data; nothing is changed here. The fixes are made in section 7, using the summary table at the end of this section.

In [14]:
import plotly.express as px   # also imported in section 4; repeated so section 5 runs on its own

print(df_transport.shape)
print(df_transport.columns.tolist())


(13754, 23)
['Collision ID', 'Crash Date', 'Crash Time', 'Crash Police Precinct', 'Crash Borough', 'Total Non-Severe Injuries', 'Total Severe Injuries', 'Total Fatalities', 'Latitude', 'Longitude', 'On Street', 'Cross Street', 'Address', 'Highway Reference Marker', 'ZipCode', 'Similar Fatalities & Severe Injuries (3Y Lookback)', 'Similar Fatalities & All Injuries (3Y Lookback)', 'Community Board', 'Council District', 'Building Identification Number (BIN)', 'Borough Block Lot (BBL)', 'Census Tract (2020)', 'Neighborhood Tabulation Area (NTA) (2020)']


SIRTA has 13,754 rows and 23 columns, well above the brief's minimum of 9. Three columns were hidden in the portal preview: `On Street`, `Cross Street` and `Address`. `On Street` matters for Q8, which groups crashes into street corridors.

In [15]:
missing_sirta = pd.DataFrame({
    "dtype": df_transport.dtypes,
    "missing": df_transport.isna().sum(),
    "missing_%": (df_transport.isna().mean() * 100).round(1),
})
missing_sirta.sort_values("missing_%", ascending=False)


,dtype,missing,missing_%
Highway Reference Marker,object,12702,92.4
Address,object,11177,81.3
Building Identification Number (BIN),float64,10967,79.7
Borough Block Lot (BBL),float64,10967,79.7
Neighborhood Tabulation Area (NTA) (2020),object,9886,71.9
Council District,float64,9883,71.9
Community Board,float64,9883,71.9
Census Tract (2020),float64,9886,71.9
ZipCode,float64,5682,41.3
Cross Street,object,5642,41.0


**What the table shows.** The area codes (Community Board, Council District, Census Tract, NTA) are missing for about 72% of rows, which is one reason we don't join on them (D3). `ZipCode`, our main join key, is missing for 41.3%.

**444 rows hold no crash.** Collision ID, date, time, police precinct and the injury counts are missing in exactly the same 444 rows. The next cell checks what those rows do contain.

In [16]:
# Rows with no Collision ID: which columns are filled in them?
no_id = df_transport["Collision ID"].isna()
print("Rows with no Collision ID:", no_id.sum())
print(df_transport[no_id].notna().sum()[lambda counts: counts > 0], "\n")

# Every other row describes a crash. The checks below use only these rows
# (nothing is removed from df_transport; cleaning happens in section 7)
sirta_crashes = df_transport[~no_id]
print("Rows with a Collision ID:", len(sirta_crashes))


Rows with no Collision ID: 444
Community Board                              444
Council District                             444
Building Identification Number (BIN)         433
Borough Block Lot (BBL)                      433
Census Tract (2020)                          444
Neighborhood Tabulation Area (NTA) (2020)    444
dtype: int64 

Rows with a Collision ID: 13310


Those 444 rows carry only area codes: there is no date, place or victim to count, so cleaning drops them (5.x issue 1). From here on, the checks use the 13,310 rows that have a Collision ID.

In [17]:
# Borough: values, and the crashes where it is missing
print(sirta_crashes["Crash Borough"].value_counts(dropna=False), "\n")

# Police precinct numbers follow borough ranges, so they can show where a crash happened
print(sirta_crashes.groupby("Crash Borough")["Crash Police Precinct"].agg(["min", "max", "nunique"]), "\n")
no_borough = sirta_crashes["Crash Borough"].isna()
print("Precincts of crashes with no borough:", sirta_crashes.loc[no_borough, "Crash Police Precinct"].unique())
sirta_crashes.loc[no_borough, ["Crash Police Precinct", "On Street", "Cross Street"]].head(8)


Crash Borough
Brooklyn         3954
Queens           3282
Manhattan        2733
Bronx            2617
Staten Island     669
NaN                55
Name: count, dtype: int64 

                 min    max  nunique
Crash Borough                       
Bronx           40.0   52.0       12
Brooklyn        60.0   94.0       23
Manhattan        1.0   34.0       22
Queens         100.0  115.0       16
Staten Island  120.0  123.0        4 

Precincts of crashes with no borough: [116.]


,Crash Police Precinct,On Street,Cross Street
7692,116.0,LAURELTON PARKWAY,NaN
7720,116.0,145 AVE,241 ST
7973,116.0,BELT PARKWAY,NaN
8008,116.0,BROOKVILLE BLVD,S CONDUIT AVE
8051,116.0,244 ST,MERRICK BLVD
8073,116.0,LAURELTON PARKWAY,NaN
8110,116.0,145 AVE,249 ST
8156,116.0,BELT PKWY,150 ST


Every crash without a borough is in police precinct 116, and precinct numbers never overlap between boroughs. Precinct 116 doesn't appear in the Queens row only because none of its crashes has a borough. The streets (Laurelton Parkway, 145 Ave, Brookville Blvd, Merrick Blvd) are in southeast Queens, so the borough can be filled as Queens (5.x issue 3).

In [18]:
# Dates are text (MM/DD/YYYY); check every one parses and the range
crash_date = pd.to_datetime(sirta_crashes["Crash Date"], format="%m/%d/%Y")
print("Crashes from", crash_date.min().date(), "to", crash_date.max().date())
print(crash_date.dt.year.value_counts().sort_index(), "\n")

# Crash Time is text: "HH:MM", or "H:MM" for single-digit hours
print("Time text lengths:", sirta_crashes["Crash Time"].str.len().value_counts().to_dict())
crash_time = pd.to_datetime(sirta_crashes["Crash Time"].str.strip(), format="%H:%M")

# Is 00:00 a real time or a stand-in for "time unknown"? Compare it with nearby minutes
minute_counts = crash_time.dt.strftime("%H:%M").value_counts()
print("Crashes at:", {t: int(minute_counts.get(t, 0)) for t in ["23:59", "00:00", "00:01", "00:30", "01:00", "12:00"]})
print("Most common exact times:", minute_counts.head(5).to_dict())


Crashes from 2022-01-01 to 2026-06-30
Crash Date
2022    2879
2023    3016
2024    3116
2025    2947
2026    1352
Name: count, dtype: int64 

Time text lengths: {5: 12654, 4: 656}
Crashes at: {'23:59': 4, '00:00': 227, '00:01': 11, '00:30': 24, '01:00': 36, '12:00': 54}
Most common exact times: {'00:00': 227, '15:00': 99, '18:00': 87, '17:00': 85, '13:00': 84}


Every date and time parses, and the dates cover exactly our window (1 Jan 2022 – 30 Jun 2026). 2026 has only six months, so it has about half the crashes of a full year.

00:00 is the most common time of all: 227 crashes, against 11 at 00:01 and 36 at 01:00. A real midnight peak would also show in the minutes around it, so 00:00 most likely means the time was not recorded (5.x issue 4).

In [19]:
# Crashes by hour of day
hours = crash_time.dt.hour.value_counts().sort_index()
px.bar(x=hours.index, y=hours.values,
       labels={"x": "Hour of day (0 = midnight)", "y": "Serious crashes"},
       title="Serious crashes by hour: the midnight bar is raised by crashes logged at 00:00").show()

# Crashes per month across the window
months = crash_date.dt.to_period("M").dt.to_timestamp().value_counts().sort_index()
px.line(x=months.index, y=months.values,
        labels={"x": "Month", "y": "Serious crashes"},
        title="Serious crashes per month, Jan 2022 – Jun 2026").show()


**By hour:** serious crashes peak in the late afternoon and early evening (849 at 17:00, 834 at 18:00) and are fewest before dawn (266 at 05:00). The midnight bar (656) is higher than 01:00 (355) partly because of the 227 crashes logged at 00:00, so Q3 must leave those out.

**By month:** the line rises and falls with the seasons: May–October have more serious crashes than January–February. Questions that compare months (Q2, Q6) should therefore compare like with like, for example the same months or a baseline group.

In [20]:
injury_cols = ["Total Non-Severe Injuries", "Total Severe Injuries", "Total Fatalities"]
print(sirta_crashes[injury_cols].describe(), "\n")

# IQR fences: medians are 0 or 1, so the fence flags almost every crash with
# more than one victim. IQR does not work for small counts like these.
for col in injury_cols:
    q1, q3 = sirta_crashes[col].quantile([0.25, 0.75])
    fence = q3 + 1.5 * (q3 - q1)
    print(f"{col}: IQR fence {fence}, crashes above it {(sirta_crashes[col] > fence).sum()}")

# Domain rules instead: no negative counts, and every SIRTA crash must have
# at least one severe injury or death (that is what SIRTA records)
print("\nNegative counts:", (sirta_crashes[injury_cols] < 0).sum().sum())
no_harm = (sirta_crashes["Total Severe Injuries"] == 0) & (sirta_crashes["Total Fatalities"] == 0)
print("Crashes with no severe injury and no death:", no_harm.sum())

# The largest crashes, to judge whether they look real
sirta_crashes.nlargest(5, "Total Non-Severe Injuries")[["Crash Date", "Crash Borough", "On Street"] + injury_cols]


       Total Non-Severe Injuries  Total Severe Injuries  Total Fatalities
count               13310.000000           13310.000000      13310.000000
mean                    0.270473               1.011495          0.082795
std                     0.836031               0.451514          0.294304
min                     0.000000               0.000000          0.000000
25%                     0.000000               1.000000          0.000000
50%                     0.000000               1.000000          0.000000
75%                     0.000000               1.000000          0.000000
max                    19.000000               7.000000          5.000000 

Total Non-Severe Injuries: IQR fence 0.0, crashes above it 2172
Total Severe Injuries: IQR fence 1.0, crashes above it 824
Total Fatalities: IQR fence 0.0, crashes above it 1047

Negative counts: 0
Crashes with no severe injury and no death: 0


,Crash Date,Crash Borough,On Street,Total Non-Severe Injuries,Total Severe Injuries,Total Fatalities
2439,01/02/2023,Manhattan,BROADWAY,19.0,2.0,0.0
12970,06/29/2026,Queens,LONG ISLAND EXPRESSWAY,18.0,0.0,3.0
3957,05/28/2023,Brooklyn,ESSEX STREET,15.0,0.0,1.0
7120,10/21/2024,Brooklyn,OCEAN PARKWAY,15.0,2.0,0.0
5967,02/27/2024,Brooklyn,NaN,14.0,1.0,0.0


The IQR rule doesn't work here. With medians of 0 or 1, its fences (0, 1 and 0) flag every crash with more than one victim: 824 to 2,172 crashes per column. Domain rules suit counts better, and none is broken: no negative counts, and every crash has at least one severe injury or death. The biggest crashes (up to 19 non-severe injuries) are multi-victim crashes, several on major roads such as Broadway and the Long Island Expressway. That is plausible, so all of them stay (5.x issue 10).

In [21]:
lookback_cols = ["Similar Fatalities & Severe Injuries (3Y Lookback)",
                 "Similar Fatalities & All Injuries (3Y Lookback)"]
print(sirta_crashes[lookback_cols].describe(), "\n")

# Where is the severe-injury lookback missing? (Q4 relies on it)
no_lookback = sirta_crashes[lookback_cols[0]].isna()
print("Missing severe-injury lookback:", no_lookback.sum())
print("By crash year:", crash_date[no_lookback].dt.year.value_counts().sort_index().to_dict())

# Locations with the highest all-injury lookback
sirta_crashes.nlargest(3, lookback_cols[1])[["Crash Borough", "On Street", "Highway Reference Marker", lookback_cols[1]]]


       Similar Fatalities & Severe Injuries (3Y Lookback)  \
count                                       11917.000000    
mean                                            0.095494    
std                                             0.421729    
min                                             0.000000    
25%                                             0.000000    
50%                                             0.000000    
75%                                             0.000000    
max                                             9.000000    

       Similar Fatalities & All Injuries (3Y Lookback)  
count                                     13310.000000  
mean                                          1.544328  
std                                           5.857394  
min                                           0.000000  
25%                                           0.000000  
50%                                           0.000000  
75%                                           1.000

,Crash Borough,On Street,Highway Reference Marker,Similar Fatalities & All Injuries (3Y Lookback)
12449,Brooklyn,VERRAZANO BRIDGE UPPER,NaN,296.0
5911,Brooklyn,VERRAZANO BRIDGE UPPER,NaN,248.0
1584,Brooklyn,VERRAZANO BRIDGE UPPER,NaN,179.0


The severe-injury lookback is missing only for crashes in 2023 and 2024; every other year is complete. The data doesn't say why. Filling these gaps with 0 would wrongly mark the crashes as "not at a repeat location", so Q4 should use only crashes with a value (5.x issue 9). The highest all-injury lookback values are on the Verrazzano Bridge, which has no highway marker (see the highway check at the end of this section).

In [22]:
no_zip = sirta_crashes["ZipCode"].isna()
has_coords = sirta_crashes["Latitude"].notna()
is_highway = sirta_crashes["Highway Reference Marker"].notna()

print("Crashes with no ZIP:", no_zip.sum())
print("  ...but with coordinates (ZIP can be filled):", (no_zip & has_coords).sum())
print("  ...and no coordinates either:", (no_zip & ~has_coords).sum())
print("Highway crashes (marker filled):", is_highway.sum(), "| of them with no ZIP:", (is_highway & no_zip).sum(), "\n")

# ZIP and coordinate coverage by year: is the gap spread evenly?
crash_year = crash_date.dt.year
coverage = pd.DataFrame({
    "crashes": crash_year.value_counts().sort_index(),
    "ZIP filled %": (~no_zip).groupby(crash_year).mean().mul(100).round(1),
    "coordinates filled %": has_coords.groupby(crash_year).mean().mul(100).round(1),
})
print(coverage, "\n")

# ZIP values: all 5 digits? any that are not postal ZIPs?
zips = sirta_crashes["ZipCode"].dropna().astype(int)
print("Distinct ZIPs:", zips.nunique(), "| lowest", zips.min(), "| highest", zips.max())

# Coordinates inside the same rough NYC box used for housing (lat 40.4–41.0, lon -74.3 to -73.6)
lat, lon = sirta_crashes["Latitude"], sirta_crashes["Longitude"]
print("Outside NYC box:", (has_coords & ~(lat.between(40.4, 41.0) & lon.between(-74.3, -73.6))).sum())

px.bar(coverage.reset_index(names="year"), x="year", y=["ZIP filled %", "coordinates filled %"],
       barmode="group", labels={"value": "% of crashes", "variable": ""},
       title="Only 16% of 2022 crashes have a ZIP code, but most have coordinates").show()


Crashes with no ZIP: 5238
  ...but with coordinates (ZIP can be filled): 4091
  ...and no coordinates either: 1147
Highway crashes (marker filled): 1052 | of them with no ZIP: 1049 

            crashes  ZIP filled %  coordinates filled %
Crash Date                                             
2022           2879          16.3                  89.0
2023           3016          68.1                  81.5
2024           3116          70.0                  66.2
2025           2947          80.3                  88.1
2026           1352          74.0                  87.6 

Distinct ZIPs: 187 | lowest 10000 | highest 11694
Outside NYC box: 0


5,238 crashes have no ZIP, but 4,091 of them have coordinates, so their ZIP can be filled in section 8. The gap is not spread evenly: in 2022 only 16.3% of crashes have a ZIP, against 68–80% in later years, while most crashes have coordinates in every year. Without filling, Q7 and Q10, which start from 2022, would rest on a small and uneven share of crashes (5.x issue 6). Almost all highway crashes (1,049 of 1,052) have no ZIP. The next cell looks at one ZIP value that is not a postal ZIP.

In [23]:
# ZIP 10000 is not a postal ZIP. Where are these crashes?
zip_10000 = sirta_crashes[sirta_crashes["ZipCode"] == 10000]
print("Crashes with ZIP 10000:", len(zip_10000))
print("  ...with coordinates:", zip_10000["Latitude"].notna().sum())
print("  ...with a street name:", zip_10000["On Street"].notna().sum())
print(zip_10000.loc[zip_10000["On Street"].notna(), ["On Street", "Cross Street"]], "\n")

# Map the ones with coordinates (hover shows the date and street, where known)
located = zip_10000.dropna(subset=["Latitude", "Longitude"])
px.scatter_map(located, lat="Latitude", lon="Longitude", hover_data=["Crash Date", "On Street"],
               zoom=13, map_style="open-street-map",
               title="The ZIP-10000 crashes that have coordinates are in the Central Park area").show()


Crashes with ZIP 10000: 44
  ...with coordinates: 6
  ...with a street name: 2
       On Street                Cross Street
3129  WEST DRIVE  TRANSVERSE ROAD NUMBER ONE
8942     EAST DR            97 ST TRANSVERSE 



Only 6 of the 44 crashes with ZIP 10000 have coordinates, and on the map those 6 fall in the Central Park area. Two of the 44 name park drives (West Drive, East Drive). For the rest we have no location, so we can't say where they happened. Cleaning keeps 10000 as its own area and states this; it will match no housing ZIP (5.x issue 7).

In [24]:
# Duplicates: fully identical rows, and Collision IDs used more than once
print("Fully duplicated rows (all rows):", df_transport.duplicated().sum())
print("Fully duplicated rows (crash rows only):", sirta_crashes.duplicated().sum())

repeated_id = sirta_crashes["Collision ID"].duplicated(keep=False)
print("Collision IDs used twice:", sirta_crashes.loc[repeated_id, "Collision ID"].nunique())
print("Distinct crashes:", sirta_crashes["Collision ID"].nunique())

# Show the repeated ID side by side to see which columns differ
sirta_crashes[repeated_id].T


Fully duplicated rows (all rows): 3
Fully duplicated rows (crash rows only): 0
Collision IDs used twice: 1
Distinct crashes: 13309


,2157,10709
Collision ID,4597666.0,4597666.0
Crash Date,01/13/2023,01/13/2026
Crash Time,21:01,21:01
Crash Police Precinct,44.0,44.0
Crash Borough,Bronx,Bronx
Total Non-Severe Injuries,0.0,0.0
Total Severe Injuries,1.0,1.0
Total Fatalities,0.0,1.0
Latitude,NaN,NaN
Longitude,NaN,NaN


The 3 fully duplicated rows are all among the 444 empty rows, which cleaning drops anyway; no crash row is repeated in full. One Collision ID (4597666) appears twice: same time, precinct and street, but dated exactly three years apart (13 Jan 2023 and 13 Jan 2026), and only the 2026 row has a death. It looks like one crash recorded twice, with the wrong year in one of the rows; cleaning keeps one row (5.x issue 2).

In [25]:
streets = sirta_crashes["On Street"].dropna()
print("Distinct street names:", streets.nunique())
print(streets.value_counts().head(10), "\n")

# The same street is written in different ways; Q8 groups crashes by street,
# so these must be made the same in cleaning
print("Crashes with a street name not in upper case:", (streets != streets.str.upper()).sum())
print("Crashes with double spaces in the street name:", streets.str.contains("  ").sum())
upper = streets.str.upper()
for short, full in [("AVE", "AVENUE"), ("ST", "STREET"), ("BLVD", "BOULEVARD"),
                    ("PKWY", "PARKWAY"), ("EXPY", "EXPRESSWAY")]:
    short_pattern, full_pattern = rf"\b{short}\b", rf"\b{full}\b"   # \b = whole word only
    print(f"{short:>4}: {upper.str.contains(short_pattern).sum():>5} crashes | "
          f"{full}: {upper.str.contains(full_pattern).sum():>5} crashes")

# "ST" means Street at the end of a name but Saint at the start (ST NICHOLAS AVE),
# so cleaning may only expand it when it is the last word
print("\nST as the first word (Saint):", upper.str.match(r"ST\b").sum(),
      "| ST as the last word (Street):", upper.str.contains(r"\bST$").sum())
print("Examples of ST as the first word:", upper[upper.str.match(r"ST\b")].value_counts().head(5).to_dict())

# Highway-type names without a Highway Reference Marker: the marker alone
# misses some highway crashes (needed for D15, which excludes highways from Q8 and Q11)
highway_name = upper.str.contains(r"\b(?:EXPRESSWAY|EXPY|EXPWY|PKWY|PARKWAY|BRIDGE|TUNNEL|THRUWAY|HWY|HIGHWAY)\b")
no_marker = sirta_crashes.loc[streets.index, "Highway Reference Marker"].isna()
print("\nCrashes on highway-type names with no marker:", (highway_name & no_marker).sum())
print(upper[highway_name & no_marker].value_counts().head(10))


Distinct street names: 2952
On Street
BROADWAY                      167
BELT PARKWAY                  143
MAJOR DEEGAN EXPRESSWAY        98
LONG ISLAND EXPRESSWAY         96
FDR DRIVE                      91
GRAND CENTRAL PKWY             91
BROOKLYN QUEENS EXPRESSWAY     78
CROSS ISLAND PARKWAY           77
3 AVENUE                       67
HENRY HUDSON PARKWAY           66
Name: count, dtype: int64 

Crashes with a street name not in upper case: 84
Crashes with double spaces in the street name: 1
 AVE:  1281 crashes | AVENUE:  2745 crashes
  ST:  1153 crashes | STREET:  1938 crashes
BLVD:   290 crashes | BOULEVARD:   512 crashes
PKWY:   231 crashes | PARKWAY:   569 crashes
EXPY:    94 crashes | EXPRESSWAY:   495 crashes



ST as the first word (Saint):

 20 | ST as the last word (Street): 1130
Examples of ST as the first word: {'ST NICHOLAS AVE': 9, 'ST ANNS AVE': 4, 'ST JOHNS PL': 4, 'ST NICHOLAS AVNEUE': 1, 'ST MARKS PLACE': 1}

Crashes on highway-type names with no marker: 679
On Street
EASTERN PARKWAY           40
TRIBOROUGH BRIDGE         30
OCEAN PARKWAY             25
KINGS HIGHWAY             22
CROSS ISLAND PARKWAY      21
EASTERN PKWY              20
LONG ISLAND EXPRESSWAY    19
VERRAZANO BRIDGE UPPER    17
SHORE PARKWAY             16
OCEAN PKWY                16
Name: count, dtype: int64


**Street names.** There are 2,952 distinct names, but many are the same street written differently (`AVE`/`AVENUE`, `PKWY`/`PARKWAY`, ...), and some have typos (e.g. `ST NICHOLAS AVNEUE`). Q8 groups crashes by street, so cleaning must give each street one name. `ST` needs care: as the last word it means Street (1,130 crashes), but as the first word it means Saint (20 crashes, e.g. ST NICHOLAS AVE). So `ST` may only be expanded when it is the last word (5.x issue 12).

**Highways.** Up to 679 more crashes are on highway-type street names without a marker. But the same word search also catches residential streets (Eastern Parkway, Ocean Parkway and Kings Highway alone account for at least 123 of them), so the real number of missed highway crashes is lower. Section 7 should flag highways from the marker plus a written list of limited-access roads, not a word search (5.x issue 8).

### 5.x Summary: issues found and what cleaning should do

| # | Issue found | Evidence | Suggested handling | Affects |
|---|---|---|---|---|
| 1 | Rows with no crash in them | 444 rows (3.2%) have no Collision ID, date, time, precinct, borough or injury counts. Only Community Board, Council District, Census Tract, NTA (and BIN/BBL in 433) are filled; 3 of these rows are exact copies of another empty row | Drop. There is no crash to count and nothing to fill the values from | All questions |
| 2 | One Collision ID used twice | ID 4597666: same time (21:01), precinct and street (West 170 Street / Plaza Dr), but dated 13 Jan 2023 and 13 Jan 2026; the 2026 row has 1 death and a lookback of 0 | Keep one row. Decide which date is right (e.g. compare with the IDs of other crashes from those months). This leaves 13,309 distinct crashes | Crash and death counts |
| 3 | Borough missing | 55 crashes, all in police precinct 116. Precinct ranges never overlap between boroughs, and the streets (Laurelton Pkwy, 145 Ave, Brookville Blvd, Merrick Blvd) are in southeast Queens | Fill with Queens, from the precinct (domain rule) | Q12, borough filter |
| 4 | 00:00 used for unknown times | 227 crashes at exactly 00:00, against 11 at 00:01 and 36 at 01:00; 00:00 is the most common time of all | Keep the crashes, but mark their hour as unknown so the hour × weekday heatmap leaves them out | Q3 |
| 5 | Dates and times stored as text | Dates `MM/DD/YYYY`, all parse; range 1 Jan 2022 – 30 Jun 2026. Times are `HH:MM`, or `H:MM` for 656 single-digit hours; all parse with `%H:%M` | Convert; build one `crash_datetime` plus `year`, `hour`, `weekday` | All time questions |
| 6 | ZIP missing | 5,238 of 13,310 crashes (39.4%); 4,091 of them have coordinates, 1,147 have neither. Only 16.3% of 2022 crashes have a ZIP, against 68–80% later | Fill ZIPs from coordinates in section 8. This is essential: Q7 and Q10 use 2022 as their starting point. The 1,147 without coordinates can be used at borough level only | All ZIP questions |
| 7 | ZIP stored as a decimal, and one false ZIP | `ZipCode` is `float64`; 187 distinct values. 44 crashes have ZIP 10000, which is not a postal ZIP. Only 6 of them have coordinates, and those fall in the Central Park area; 2 name park drives. The other crashes have no location | Convert to 5-character text. Keep 10000 as its own area (it matches no housing ZIP), and say so | Join with housing |
| 8 | Highway crashes not fully marked | Highway Reference Marker is filled for 1,052 crashes; 1,049 of them have no ZIP. Up to 679 more crashes are on highway-type street names without a marker (e.g. Cross Island Pkwy, Triborough Bridge, Long Island Expressway), but the word search also catches residential streets (Eastern Parkway, Ocean Parkway, Kings Highway: at least 123 crashes), so the real number is lower | Highway flag = marker filled, or street on a written list of limited-access roads (not a word search) | Q5, Q8, Q11 (D12, D15) |
| 9 | Repeat-location lookback missing | Severe-injury lookback is missing for 1,393 crashes, all from 2023 (766) and 2024 (627) | Do not fill with 0 (that would mean "not a repeat location"). Q4 uses only crashes with a value and states the gap | Q4 |
| 10 | IQR does not suit injury counts | Medians are 0 or 1, so the IQR fences (0, 1, 0) flag 824–2,172 crashes, i.e. every crash with more than one victim | Use domain rules instead. None are broken: no negative counts, and every crash has at least one severe injury or death. The largest crash (19 non-severe injuries, Broadway, 2 Jan 2023) is plausible. Keep all | Q5, totals |
| 11 | Very high lookback values | All-injury lookback reaches 296, on the Verrazzano Bridge | Keep: a busy bridge with many past crashes is plausible | Q4 |
| 12 | Street names written in different ways | 2,952 distinct names; both `AVE` and `AVENUE`, `ST` and `STREET`, `PKWY` and `PARKWAY`, etc.; typos (e.g. `AVNEUE`); 84 crashes with a name not in upper case; 1 with a double space. `ST` is Street as the last word (1,130 crashes) but Saint as the first word (20, e.g. ST NICHOLAS AVE) | Upper case, trim spaces, expand abbreviations so each corridor has one name. Expand `ST` only when it is the last word | Q8 |
| 13 | IDs stored as decimals | Collision ID and Crash Police Precinct are `float64` because of the 444 empty rows | Convert to whole numbers after dropping those rows | Duplicate checks |
| 14 | Columns no question uses | Address (81.3% missing), BIN and BBL (79.7%), Community Board, Council District, Census Tract and NTA (71.9%; 2020 codes that don't match housing) | Drop (see D3) | — |

**Checked and fine:** every date and time parses; all coordinates fall inside NYC; Borough has 5 clean values; no crash row is fully duplicated; no negative injury counts; every crash has at least one severe injury or death.

## 6. Housing cleaning
> **Owner:** Mostafa Alaa

This section fixes the housing issues listed in table 4.x. Each step shows the check, the decision with its reason, and the alternatives we rejected. The raw `df_housing` is not changed. The result is `housing_clean`: one row per building record, with the column names from `docs/data_dictionary.md`. Section 8 uses it for integration.

**Main rule:** we only drop something if it is unusable. A building without a ZIP still counts at borough level, and an unfinished building still counts as "under construction" (Q6). Each question then picks the rows it needs, using the flags added here.

In [26]:
# Work on a copy, so df_housing stays the raw data for the before/after check (6.6)
housing = df_housing.copy()
print("Raw housing data:", housing.shape)

# Column groups used below
income_cols = ["Extremely Low Income Units", "Very Low Income Units", "Low Income Units",
               "Moderate Income Units", "Middle Income Units", "Other Income Units"]
bedroom_cols = ["Studio Units", "1-BR Units", "2-BR Units", "3-BR Units",
                "4-BR Units", "5-BR Units", "6-BR+ Units", "Unknown-BR Units"]
tenure_cols = ["Counted Rental Units", "Counted Homeownership Units"]
breakdown_cols = income_cols + bedroom_cols + tenure_cols

# Confidential projects hide their name and address (4.x issue 1); flag them once
housing["is_confidential"] = housing["Project Name"].eq("CONFIDENTIAL")
print("Confidential rows:", housing["is_confidential"].sum())

# Dates are parsed here only for checks; they are converted for good in 6.4
completion_check = pd.to_datetime(housing["Building Completion Date"], format="%m/%d/%Y")
start_check = pd.to_datetime(housing["Project Start Date"], format="%m/%d/%Y")
project_done_check = pd.to_datetime(housing["Project Completion Date"], format="%m/%d/%Y")


Raw housing data: (9551, 41)
Confidential rows: 1949


`housing` is the working copy (9,551 rows, 41 columns, 1,949 of them confidential). The three `_check` date columns are only used by the checks in 6.1–6.3; the real conversion happens in 6.4.

### 6.1 Missing values

In [27]:
# Issue 2: confidential rows use "----" instead of a house number and street.
# These are missing values in disguise, so make them real missing values.
for col in ["Number", "Street"]:
    housing[col] = housing[col].mask(housing[col].eq("----"))
print("Missing after replacing '----':", housing[["Number", "Street"]].isna().sum().to_dict())
print("...exactly the confidential rows:", housing["Street"].isna().equals(housing["is_confidential"]), "\n")

# Issue 3: do empty unit-breakdown cells mean 0? Test each group: with empty = 0,
# does the group add up to All Counted Units?
known_total = housing["All Counted Units"].notna()
for group, cols in [("Income tiers", income_cols), ("Bedroom sizes", bedroom_cols), ("Tenure", tenure_cols)]:
    group_sum = housing.loc[known_total, cols].fillna(0).sum(axis=1)
    adds_up = group_sum.eq(housing.loc[known_total, "All Counted Units"]).sum()
    print(f"{group}: adds up in {adds_up:,} of {known_total.sum():,} rows; "
          f"cells written as 0: {(housing[cols] == 0).sum().sum()}")

# Every group adds up and the data never writes a 0, so an empty cell means "0 units of this kind"
housing[breakdown_cols] = housing[breakdown_cols].fillna(0)
print("\nEmpty breakdown cells left:", housing[breakdown_cols].isna().sum().sum())


Missing after replacing '----': {'Number': 1949, 'Street': 1949}
...exactly the confidential rows: True 

Income tiers: adds up in 9,522 of 9,522 rows; cells written as 0: 0
Bedroom sizes: adds up in 9,522 of 9,522 rows; cells written as 0: 0
Tenure: adds up in 9,522 of 9,522 rows; cells written as 0: 0

Empty breakdown cells left: 0


**`----` placeholders (issue 2) → missing.** All 1,949 placeholders were in the confidential rows. They now count as missing, so no later step can mistake `----` for a street name.

**Empty unit-breakdown cells (issue 3) → 0.** With empty = 0, the income tiers, the bedroom sizes and the tenure columns each add up exactly to All Counted Units in all 9,522 rows with a known total, and no cell anywhere is written as 0. So an empty cell means "no units of this kind".
*Alternatives rejected:* leaving them empty would make Q9's shares (e.g. % of units that are rental) impossible to compute; filling with a typical value such as the median would invent units and break the totals.

In [28]:
# Issue 4: All Counted Units is missing in some rows. Can it be rebuilt from the breakdowns?
no_units = housing["All Counted Units"].isna()
print("Rows without All Counted Units:", no_units.sum())
print("Breakdown total in those rows:", housing.loc[no_units, breakdown_cols].sum(axis=1).unique())
print("Completed:", completion_check[no_units].notna().sum(),
      "| completed 1 Jan 2022 – 30 Jun 2026:", completion_check[no_units].between("2022-01-01", "2026-06-30").sum())
print("Their Total Units (affordable + market-rate):", housing.loc[no_units, "Total Units"].sum())
housing.loc[no_units, ["Project Name", "Borough", "Reporting Construction Type",
                       "Total Units", "Building Completion Date"]].head(10)


Rows without All Counted Units: 29
Breakdown total in those rows: [0.]
Completed: 18 | completed 1 Jan 2022 – 30 Jun 2026: 1
Their Total Units (affordable + market-rate): 2525


,Project Name,Borough,Reporting Construction Type,Total Units,Building Completion Date
185,67 4TH AVENUE,Brooklyn,New Construction,10,NaN
1764,261 & 315 GRAND CONCOURSE,Bronx,New Construction,283,NaN
2292,325 BOND STREET,Brooklyn,New Construction,425,NaN
3932,2916 WEST 16TH STREET,Brooklyn,New Construction,324,NaN
4668,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4670,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4671,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4672,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4721,ROSE CASTLE,Brooklyn,New Construction,122,NaN
5198,LINDEN PLAZA PRESERVATION LP.YR15.FY20,Brooklyn,Preservation,32,12/27/2019


**Missing All Counted Units (issue 4) → kept as missing.** All 29 rows also have empty breakdowns (total 0), so there is nothing to rebuild the number from. Only 1 of them was completed inside our window, so they barely affect unit totals. Pandas sums skip missing values, so these rows simply add nothing to unit totals.
*Alternatives rejected:* filling with Total Units would count market-rate homes as affordable (these 29 rows have 2,525 Total Units); filling with 0 would say the building has no affordable units although it is in an affordable-housing programme; dropping the rows would lose buildings whose size (Total Units) Q2 and Q6 can still use.

In [29]:
# Issue 5: buildings with no completion date. Unfinished, or is the date lost?
not_completed = completion_check.isna()
print("No building completion date:", not_completed.sum())
print("  ...and no project completion date either:", (not_completed & project_done_check.isna()).sum())
print("  ...confidential:", (not_completed & housing["is_confidential"]).sum())
print("Project start year of these buildings:")
print(start_check[not_completed].dt.year.value_counts().sort_index().to_dict())


No building completion date: 1666
  ...and no project completion date either: 1653
  ...confidential: 70
Project start year of these buildings:
{2014: 1, 2015: 1, 2016: 8, 2017: 16, 2018: 89, 2019: 9, 2020: 33, 2021: 86, 2022: 82, 2023: 212, 2024: 376, 2025: 430, 2026: 323}


**Missing completion date (issue 5) → kept as "not completed yet".** 1,653 of the 1,666 buildings also have no project completion date, and most of their projects started recently (1,341 in 2023–2026). So these are mostly buildings still in progress, not lost dates. They stay with `is_completed = False` (6.4): they don't count as completions (Q1, Q2, Q7, Q12), but Q6 counts them as under construction and Q10 counts their project starts.
*Alternative rejected:* guessing a completion date (e.g. start date + a typical build time) would invent the very events that the before/after crash comparisons depend on.

In [30]:
# Issue 1: buildings with no ZIP or coordinates
no_location = housing["Postcode"].isna()
print("No ZIP:", no_location.sum(), "| also no coordinates:", (no_location & housing["Latitude"].isna()).sum())
print("  ...confidential:", (no_location & housing["is_confidential"]).sum(),
      "| not confidential:", (no_location & ~housing["is_confidential"]).sum())
print("Missing Building ID:", housing["Building ID"].isna().sum(),
      "| all of them confidential:", housing.loc[housing["Building ID"].isna(), "is_confidential"].all())

# The non-confidential rows without a location
housing.loc[no_location & ~housing["is_confidential"],
            ["Project Name", "Number", "Street", "Borough", "Reporting Construction Type",
             "Building Completion Date", "Total Units"]]


No ZIP: 1958 | also no coordinates: 1958
  ...confidential: 1949 | not confidential: 9
Missing Building ID: 1949 | all of them confidential: True


,Project Name,Number,Street,Borough,Reporting Construction Type,Building Completion Date,Total Units
467,122-21 14TH AVENUE,1422-21,14 AVENUE,Queens,New Construction,NaN,99
560,ARVERNE EAST - BUILDING D - RENTAL,3541,BEACH 36 STREET,Queens,New Construction,NaN,230
561,ARVERNE EAST OPEN DOOR,3541,BEACH 36 STREET,Queens,New Construction,NaN,90
563,75-42 GRAND CENTRAL PARKWAY,75-42,GRAND CENTRAL PARKWAY NORTH,Queens,New Construction,NaN,10
751,LE GRANDE VILLE,1701,WEST 176 STREET,Bronx,New Construction,NaN,35
769,OCEAN CREST,726,BEACH CHANNEL DRIVE,Queens,New Construction,NaN,89
1980,470 KENT AVENUE,11,WHARF WAY,Brooklyn,New Construction,NaN,183
3313,101-19 ROCKAWAY BEACH BOULEVARD,101-19,ROCKAWAY BEACH BOULEVAR,Queens,New Construction,NaN,60
6806,SOUTHEASTERN QUEENS VACANT HOMES-CLUSTER II,133-16,VAN WYCK EXPRESSWAY SR WEST,Queens,Preservation,06/19/2019,1


**No ZIP or coordinates (issue 1) → kept, borough level only.** The 1,958 rows without a ZIP are exactly the rows without coordinates. 1,949 are confidential projects, which hide their address on purpose; Building ID is missing in exactly these rows too. The other 9 have an address: 8 are New Construction buildings not yet completed, and 1 is a single-unit preservation home from 2019.
Filling their location would need an outside geocoding service, and the 8 unfinished buildings don't count as completions anyway. All 1,958 keep their borough, so they still count in Q12; `has_location` (6.4) marks them.

### 6.2 Outliers

In [31]:
units = housing["All Counted Units"]
size = housing["Total Units"]

# IQR (as in section 4): the fence flags large buildings, not errors
q1, q3 = units.quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print(f"IQR fence: {fence:.0f} affordable units | buildings above it: {(units > fence).sum():,}\n")

# Domain rules: values that cannot be true for a building in this dataset
lat, lon = housing["Latitude"], housing["Longitude"]
domain_checks = pd.Series({
    "affordable units below 1": (units < 1).sum(),
    "total units below 1": (size < 1).sum(),
    "affordable units above total units": (units > size).sum(),
    "coordinates outside NYC": (lat.notna() & ~(lat.between(40.4, 41.0) & lon.between(-74.3, -73.6))).sum(),
    "project start outside 2014 – Jun 2026": (~start_check.between("2014-01-01", "2026-06-30")).sum(),
    "completion before 2014": (completion_check < "2014-01-01").sum(),
}, name="rows breaking the rule")
print(domain_checks, "\n")

# Size groups for Q2 and Q6, by Total Units (the number of new residents matters there)
housing["size_group"] = pd.cut(size, bins=[0, 49, 199, float("inf")],
                               labels=["Small (<50)", "Medium (50-199)", "Large (200+)"])
print(housing["size_group"].value_counts().sort_index(), "\n")

# The largest buildings: do they look like real projects?
housing.nlargest(8, "Total Units")[["Project Name", "Borough", "Reporting Construction Type",
                                    "All Counted Units", "Total Units"]]


IQR fence: 72 affordable units | buildings above it: 1,249

affordable units below 1                 0
total units below 1                      0
affordable units above total units       0
coordinates outside NYC                  0
project start outside 2014 – Jun 2026    0
completion before 2014                   0
Name: rows breaking the rule, dtype: int64 

size_group
Small (<50)        7309
Medium (50-199)    1698
Large (200+)        544
Name: count, dtype: int64 



,Project Name,Borough,Reporting Construction Type,All Counted Units,Total Units
1442,25 WATER STREET,Manhattan,New Construction,330.0,1320
6902,605 WEST 42ND STREET PHASE II,Manhattan,New Construction,59.0,1175
9157,605 WEST 42ND ST,Manhattan,New Construction,235.0,1175
4533,5 POINTZ LIC,Queens,New Construction,337.0,1122
3406,12 ROCKWELL PLACE,Brooklyn,New Construction,227.0,1046
9040,606 WEST 57TH STREET,Manhattan,New Construction,258.0,1028
6687,WEST END TOWERS,Manhattan,Preservation,250.0,1000
8451,Fleet Project 43-25 Hunter Street (20150032),Queens,New Construction,195.0,974


**Outliers (issue 9) → kept; domain rules instead of IQR.** The IQR fence (72 affordable units) flags 1,249 buildings, but no building breaks a domain rule: every building has at least 1 unit, affordable units never exceed total units, all coordinates are inside NYC and all dates are in the dataset's range. The largest buildings are named large developments (e.g. 25 Water Street with 1,320 units, 5 Pointz LIC with 1,122), not typing errors.
Dropping or capping them would remove exactly the large buildings Q2 and Q6 study, and shrink the unit totals. Instead we keep them and add `size_group` by Total Units for Q2: small < 50 (7,309 buildings), medium 50–199 (1,698), large 200+ (544).

### 6.3 Inconsistencies

In [32]:
# Issue 10: buildings completed before their project started
before_start = completion_check < start_check
housing["completed_before_start"] = before_start
print("Completed before project start:", before_start.sum(),
      "| of them in 1 Jan 2022 – 30 Jun 2026:", (before_start & completion_check.between("2022-01-01", "2026-06-30")).sum())

# Other date-order checks between a building and its project
print("Building completed after its project completed:", (completion_check > project_done_check).sum())
print("Building has a completion date but its project has none:",
      (completion_check.notna() & project_done_check.isna()).sum())


Completed before project start: 20 | of them in 1 Jan 2022 – 30 Jun 2026: 1
Building completed after its project completed: 0
Building has a completion date but its project has none: 166


**Completed before the project started (issue 10) → kept and flagged.** 20 buildings are dated before their project's start, and only 1 of them falls in our window. Most gaps are small (1–20 days for 14 of them, section 4), which looks like paperwork order rather than wrong buildings, so the rows stay with `completed_before_start = True`.
No building is dated after its project's completion. 166 buildings are complete while their project isn't; that is normal for projects whose buildings finish at different times.

In [33]:
# The same building can appear in more than one project (e.g. preserved again under a new agreement)
repeated = housing["Building ID"].notna() & housing["Building ID"].duplicated(keep=False)
housing["in_several_projects"] = repeated
print("Buildings in more than one project:", housing.loc[repeated, "Building ID"].nunique(), "| rows:", repeated.sum())
combos = housing[repeated].groupby("Building ID")["Reporting Construction Type"].agg(lambda types: " + ".join(sorted(types)))
print(combos.value_counts(), "\n")

# Would counting every row count the same building's completion twice in our window?
completed_in_window = completion_check.between("2022-01-01", "2026-06-30")
twice = completed_in_window[repeated].groupby(housing.loc[repeated, "Building ID"]).sum().gt(1)
print("Buildings completed twice inside 1 Jan 2022 – 30 Jun 2026:", twice.sum(), "\n")

# Example rows, oldest project first
(housing[repeated].assign(start=start_check).sort_values(["Building ID", "start"])
 [["Building ID", "Project Name", "Reporting Construction Type", "Project Start Date",
   "Building Completion Date", "All Counted Units"]].head(6))


Buildings in more than one project: 123 | rows: 247
Reporting Construction Type
Preservation + Preservation                   107
New Construction + New Construction            13
New Construction + Preservation                 2
Preservation + Preservation + Preservation      1
Name: count, dtype: int64 

Buildings completed twice inside 1 Jan 2022 – 30 Jun 2026: 2 



,Building ID,Project Name,Reporting Construction Type,Project Start Date,Building Completion Date,All Counted Units
5521,1338.0,TAINO TOWERS.HUDMF.FY19,Preservation,06/27/2019,06/27/2019,208.0
2843,1338.0,TAINO TOWERS PHASE 3.HUDMF.FY23,Preservation,06/23/2023,NaN,208.0
5522,1339.0,TAINO TOWERS.HUDMF.FY19,Preservation,06/27/2019,06/27/2019,92.0
2844,1339.0,TAINO TOWERS PHASE 3.HUDMF.FY23,Preservation,06/23/2023,NaN,92.0
5523,2098.0,TAINO TOWERS.HUDMF.FY19,Preservation,06/27/2019,06/27/2019,208.0
2845,2098.0,TAINO TOWERS PHASE 3.HUDMF.FY23,Preservation,06/23/2023,NaN,208.0


**One building in several projects (new issue) → all rows kept, flagged.** 123 buildings appear in 2 or 3 projects (247 rows). Most are Preservation twice (107): the same building signs a new affordability agreement years later, e.g. Taino Towers in 2019 and again from 2023. Each row is a separate agreement with its own units, so every row stays for unit totals. Only 2 buildings were completed twice inside our window, so double counting there is small.
For building-level questions (Q8 map, Q11 doorstep counts) a building must count once, so section 8 uses one row per `building_id` there; `in_several_projects` marks these rows. Total Units also repeats across phases (605 West 42nd Street shows 1,175 in both phases), so building size must never be summed across rows.

In [34]:
# BIN: numbers like 1000000, 2000000, ... are placeholders for "no number yet"
# (the first digit is the borough). BIN is not used by any question, so it is dropped in 6.4.
bin_number = housing["BIN"].dropna().astype("int64")
placeholder_bin = bin_number % 1_000_000 == 0
print("Placeholder BINs:", placeholder_bin.sum(), bin_number[placeholder_bin].value_counts().to_dict())

# Two coordinate pairs: how far apart are they, and which is more complete?
both_pairs = housing["Latitude"].notna() & housing["Latitude (Internal)"].notna()
north_south_gap_m = (housing["Latitude"] - housing["Latitude (Internal)"]).abs()[both_pairs] * 111_000  # 1° latitude ≈ 111 km
print(f"\nNorth–south gap between the two pairs: median {north_south_gap_m.median():.0f} m, "
      f"largest {north_south_gap_m.max():.0f} m")
print("Missing: Latitude", housing["Latitude"].isna().sum(),
      "| Latitude (Internal)", housing["Latitude (Internal)"].isna().sum(),
      "| Latitude only:", (housing["Latitude"].notna() & housing["Latitude (Internal)"].isna()).sum(), "\n")

# Category values: any spelling variants or stray spaces?
for col in ["Borough", "Reporting Construction Type"]:
    print(f"{col}: {sorted(housing[col].unique())} | stray spaces: {(housing[col] != housing[col].str.strip()).sum()}")


Placeholder BINs: 482 {3000000: 177, 2000000: 153, 4000000: 90, 1000000: 43, 5000000: 19}

North–south gap between the two pairs: median 21 m, largest 1129 m
Missing: Latitude 1958 | Latitude (Internal) 2044 | Latitude only: 86 

Borough: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island'] | stray spaces: 0
Reporting Construction Type: ['New Construction', 'Preservation'] | stray spaces: 0


**BIN placeholders (new issue) → column dropped.** 482 BINs are placeholders (1000000 to 5000000), so BIN can't identify buildings. No question uses BIN or BBL (D3), so both are dropped in 6.4 instead of being repaired.

**Two coordinate pairs (issue 12) → keep `Latitude`/`Longitude`.** The pairs are close (median 21 m apart north–south; the largest gap is about 1.1 km), and `Latitude` is filled in 86 more rows than the Internal pair. *Alternative rejected:* averaging the two pairs would mix two methods with no gain.

**Categories** are already clean: 5 boroughs, 2 construction types, no stray spaces.

### 6.4 Standard formats

In [35]:
# Dates: text "MM/DD/YYYY" -> datetime (every non-empty value parsed in section 4)
date_cols = ["Project Start Date", "Project Completion Date", "Building Completion Date"]
for col in date_cols:
    housing[col] = pd.to_datetime(housing[col], format="%m/%d/%Y")

# ZIP: 11432.0 -> "11432". Text, because a ZIP is a label, not a quantity;
# SIRTA's ZIPs get the same format in section 7, so the two can be joined
housing["Postcode"] = housing["Postcode"].astype("Int64").astype("string")

# IDs and unit counts: decimals -> whole numbers. "Int64" (capital I) allows missing values:
# confidential rows have no Building ID, and 29 rows have no All Counted Units
for col in ["Project ID", "Building ID", "All Counted Units", "Total Units"] + breakdown_cols:
    housing[col] = housing[col].astype("Int64")

# Flags used by later sections
housing["has_location"] = housing["Latitude"].notna()
housing["is_completed"] = housing["Building Completion Date"].notna()
housing["completion_year"] = housing["Building Completion Date"].dt.year.astype("Int64")

print(housing[date_cols + ["Postcode", "Building ID", "All Counted Units"]].dtypes, "\n")
housing_zips = housing["Postcode"].dropna()
print("ZIP examples:", housing_zips.head(3).tolist(), "| all 5 characters:", housing_zips.str.len().eq(5).all())


Project Start Date          datetime64[ns]
Project Completion Date     datetime64[ns]
Building Completion Date    datetime64[ns]
Postcode                    string[python]
Building ID                          Int64
All Counted Units                    Int64
dtype: object

ZIP examples:

 ['11432', '11211', '10030'] | all 5 characters: True


**Formats (issues 6–8).** Dates are now real dates, so they can be compared and grouped by month or year. ZIP is now 5-character text (`"11432"`), the same format SIRTA's ZIP gets in section 7, so the join key matches exactly. IDs and unit counts are whole numbers; `Int64` keeps missing values missing instead of forcing decimals.

Three flags are added for later sections: `has_location`, `is_completed`, and `completion_year`, which lets the website's year filter work on buildings (gap C in the `charts.py` contract).

In [36]:
# Names: the shared data dictionary (docs/data_dictionary.md) uses short snake_case names
rename = {
    "Building ID": "building_id", "Project ID": "project_id", "Project Name": "project_name",
    "Number": "number", "Street": "street", "Borough": "borough", "Postcode": "zip",
    "Latitude": "lat", "Longitude": "lon", "Reporting Construction Type": "construction_type",
    "Project Start Date": "project_start", "Building Completion Date": "completion_date",
    "All Counted Units": "all_counted_units", "Total Units": "total_units",
    "Extremely Low Income Units": "units_extremely_low", "Very Low Income Units": "units_very_low",
    "Low Income Units": "units_low", "Moderate Income Units": "units_moderate",
    "Middle Income Units": "units_middle", "Other Income Units": "units_other_income",
    "Studio Units": "units_studio", "1-BR Units": "units_1br", "2-BR Units": "units_2br",
    "3-BR Units": "units_3br", "4-BR Units": "units_4br", "5-BR Units": "units_5br",
    "6-BR+ Units": "units_6br_plus", "Unknown-BR Units": "units_unknown_br",
    "Counted Rental Units": "units_rental", "Counted Homeownership Units": "units_homeownership",
}
flag_cols = ["completion_year", "size_group", "is_confidential", "has_location", "is_completed",
             "completed_before_start", "in_several_projects"]
housing_clean = housing.rename(columns=rename)[list(rename.values()) + flag_cols]

# Columns no question uses (4.x issues 12 and 13), plus BIN/BBL (6.3) and Project Completion Date
dropped = [col for col in df_housing.columns if col not in rename]
print("Dropped columns:", dropped)
print("\nClean table:", housing_clean.shape)
housing_clean.dtypes


Dropped columns: ['Project Completion Date', 'BBL', 'BIN', 'Community Board', 'Council District', 'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude (Internal)', 'Longitude (Internal)', 'Extended Affordability Only', 'Prevailing Wage Status']

Clean table: (9551, 37)


building_id                        Int64
project_id                         Int64
project_name                      object
number                            object
street                            object
borough                           object
zip                       string[python]
lat                              float64
lon                              float64
construction_type                 object
project_start             datetime64[ns]
completion_date           datetime64[ns]
all_counted_units                  Int64
total_units                        Int64
units_extremely_low                Int64
units_very_low                     Int64
units_low                          Int64
units_moderate                     Int64
units_middle                       Int64
units_other_income                 Int64
units_studio                       Int64
units_1br                          Int64
units_2br                          Int64
units_3br                          Int64
units_4br       

**Names and unused columns (issue 13).** Columns are renamed to the short names in `docs/data_dictionary.md`, so the integration code and the website use one set of names. 11 columns are dropped: BIN, BBL, Community Board, Council District, Census Tract and NTA (not join keys, D3); the Internal coordinates (issue 12); Extended Affordability Only and Prevailing Wage Status (no question uses them); and Project Completion Date (only needed for the checks above). `housing_clean` has 9,551 rows and 37 columns.

### 6.5 Duplicates

In [37]:
# Fully identical rows (all 41 original columns), and the same building twice in one project
print("Fully duplicated rows:", housing.drop(columns=flag_cols).duplicated().sum())
print("Repeated Project ID + Building ID:",
      housing.dropna(subset=["Building ID"]).duplicated(["Project ID", "Building ID"]).sum())

# Confidential rows have no Building ID, so they can only be compared on their other columns.
# In the clean table (fewer columns) some of them look alike: are they real copies?
confidential_clean = housing_clean[housing_clean["is_confidential"]]
look_alike = confidential_clean.duplicated(keep=False)
print("\nConfidential rows that look alike in the clean table:", look_alike.sum())
area_columns = housing.loc[look_alike[look_alike].index,
                           list(rename) + ["Community Board", "Council District", "BIN", "BBL"]]
print("...of them still identical once the dropped area columns are added back:",
      area_columns.duplicated(keep=False).sum())


Fully duplicated rows:

 0
Repeated Project ID + Building ID: 0

Confidential rows that look alike in the clean table: 0
...of them still identical once the dropped area columns are added back: 0


**Duplicates → none to remove.** No row repeats another in all 41 original columns, and no project lists the same building twice. Confidential rows can't be checked by Building ID, but none of them is identical to another even in the reduced clean table. The only repeats are buildings in several projects, which are separate agreements and stay (6.3).

### 6.6 Before/after check

In [38]:
# Before/after: each fix should show up here, and nothing else should have changed
before_after = pd.DataFrame({
    "raw (df_housing)": [
        len(df_housing), df_housing.shape[1],
        df_housing[["Number", "Street"]].eq("----").sum().sum(),
        df_housing[breakdown_cols].isna().sum().sum(),
        df_housing["All Counted Units"].isna().sum(),
        df_housing["Building Completion Date"].isna().sum(),
        df_housing["Postcode"].isna().sum(),
        str(df_housing["Postcode"].dtype), str(df_housing["Building Completion Date"].dtype),
        f"{df_housing['All Counted Units'].sum():,.0f}",
    ],
    "clean (housing_clean)": [
        len(housing_clean), housing_clean.shape[1],
        housing_clean[["number", "street"]].eq("----").sum().sum(),
        housing_clean[[rename[c] for c in breakdown_cols]].isna().sum().sum(),
        housing_clean["all_counted_units"].isna().sum(),
        housing_clean["completion_date"].isna().sum(),
        housing_clean["zip"].isna().sum(),
        str(housing_clean["zip"].dtype), str(housing_clean["completion_date"].dtype),
        f"{housing_clean['all_counted_units'].sum():,}",
    ],
}, index=["Rows", "Columns", "'----' placeholders", "Empty unit-breakdown cells",
          "Missing All Counted Units", "Missing completion date", "Missing ZIP",
          "ZIP type", "Completion date type", "Total affordable units"])
before_after


,raw (df_housing),clean (housing_clean)
Rows,9551,9551
Columns,41,37
'----' placeholders,3898,0
Empty unit-breakdown cells,106280,0
Missing All Counted Units,29,29
Missing completion date,1666,1666
Missing ZIP,1958,1958
ZIP type,float64,string
Completion date type,object,datetime64[ns]
Total affordable units,"320,946","320,946"


Every fix shows up as intended: the placeholders and empty breakdown cells are gone, and ZIP and dates have the right types. The number of rows (9,551) and the total of affordable units (320,946) are unchanged, so no building was lost and no unit was invented. The missing unit counts, completion dates and ZIPs stay missing on purpose (6.1).

In [39]:
# How many buildings each kind of question can use, after cleaning
in_window = housing_clean["completion_date"].between("2022-01-01", "2026-06-30")
located = housing_clean["has_location"]
new_construction = housing_clean["construction_type"].eq("New Construction")
usable = pd.Series({
    "All buildings (borough-level questions)": len(housing_clean),
    "With ZIP and coordinates (ZIP and distance questions)": located.sum(),
    "...and completed 1 Jan 2022 – 30 Jun 2026": (located & in_window).sum(),
    "...and New Construction (e.g. Q1)": (located & in_window & new_construction).sum(),
})
print(usable, "\n")
print(f"Affordable units completed 1 Jan 2022 – 30 Jun 2026: {housing_clean.loc[in_window, 'all_counted_units'].sum():,}"
      f" (in buildings with a ZIP: {housing_clean.loc[in_window & located, 'all_counted_units'].sum():,})")
print("Completed after 30 Jun 2026 (outside the window, kept for Q6):",
      (housing_clean["completion_date"] > "2026-06-30").sum())

fig = px.bar(x=usable.values, y=usable.index, orientation="h", text=usable.values,
             labels={"x": "Buildings", "y": ""},
             title="Buildings each kind of question can use after cleaning")
fig.update_yaxes(autorange="reversed")
fig.show()


All buildings (borough-level questions)                  9551
With ZIP and coordinates (ZIP and distance questions)    7593
...and completed 1 Jan 2022 – 30 Jun 2026                2599
...and New Construction (e.g. Q1)                        1670
dtype: int64 



Affordable units completed 1 Jan 2022 – 30 Jun 2026: 115,547 (in buildings with a ZIP: 114,249)
Completed after 30 Jun 2026 (outside the window, kept for Q6): 16


**What each kind of question can use.** All 9,551 buildings count at borough level. 7,593 have a ZIP and coordinates; 2,599 of those were completed inside our window, and 1,670 of those are New Construction, the set Q1 starts from. The 16 buildings completed after 30 Jun 2026 stay in the table: they were still under construction during our window, which matters for Q6, but they are not counted as completions.

**Check of the Story hook.** 115,547 affordable units were completed between 1 Jan 2022 and 30 Jun 2026, which matches the hook's "about 115,500" (taken from the raw export).

### 6.x Summary: housing cleaning decisions

| 4.x issue | Decision | Rows affected | Where |
|---|---|---|---|
| 1 No location | Keep; borough level only; flag `has_location` | 1,958 (1,949 confidential) | 6.1 |
| 2 `----` placeholders | Replace with missing | 1,949 rows (3,898 cells) | 6.1 |
| 3 Empty unit breakdowns | Fill with 0 (every group adds up to the total) | 106,280 cells | 6.1 |
| 4 Missing All Counted Units | Keep missing (nothing to rebuild it from) | 29 | 6.1 |
| 5 Missing completion date | Keep as not completed; flag `is_completed` | 1,666 | 6.1 |
| 6 Dates as text | Convert to dates | 3 columns | 6.4 |
| 7 ZIP as decimal | 5-character text | 7,593 ZIPs | 6.4 |
| 8 IDs as decimals | Whole numbers (`Int64`) | — | 6.4 |
| 9 Very large buildings | Keep (no domain rule broken); add `size_group` | 1,249 above the IQR fence | 6.2 |
| 10 Completed before start | Keep; flag `completed_before_start` | 20 | 6.3 |
| 11 Completions after the window | Keep (Q6 needs them); not counted as completions | 16 | 6.6 |
| 12 Two coordinate pairs | Keep `Latitude`/`Longitude`; drop the Internal pair | — | 6.3, 6.4 |
| 13 Unused columns | Drop 11 columns | — | 6.4 |
| New: building in several projects | Keep all rows; flag `in_several_projects`; one row per building for Q8 and Q11 | 123 buildings, 247 rows | 6.3 |
| New: placeholder BINs | Drop BIN (not used) | 482 | 6.3 |

No rows were dropped: `housing_clean` has all 9,551 buildings and all 320,946 affordable units.

## 7. SIRTA cleaning
> **Owner:** Ali

### 7.1 Missing values

### 7.2 Outliers

### 7.3 Inconsistencies

### 7.4 Standard formats

### 7.5 Duplicates

### 7.6 Before/after check

## 8. Integration
> **Owner:** Moustafa Maged

### 8.1 Join keys and why

### 8.2 Granularity and aggregation

### 8.3 Spatial join

## 9. Post-integration cleaning and export
> **Owner:** Moustafa Maged

### 9.1 NaNs from joins

### 9.2 Redundant columns

### 9.3 Data types

### 9.4 Export to `data/processed`

## 10. Research questions
> **Owner:** Everyone

### 10.1 Q1
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.2 Q2
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.3 Q3
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.4 Q4
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.5 Q5
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.6 Q6
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.7 Q7
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.8 Q8
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.9 Q9
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.10 Q10
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.11 Q11
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.12 Q12
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

## 11. Story summary
> **Owner:** Mostafa Ayman

### Headline findings

### Limitations

### Impact

## 12. Appendix
> **Owner:** Mohamed

### 12.1 Datasets

| Domain | Dataset | Catalog | ID | Link |
|---|---|---|---|---|
| Housing & Development | Affordable Housing Production by Building | H2 | `hg8x-zxpr` | [NYC Open Data](https://data.cityofnewyork.us/Housing-Development/Affordable-Housing-Production-by-Building/hg8x-zxpr/about_data) |
| Transportation | Serious Injury Response Tracking & Analysis Program (SIRTA) | T4 | `xeqp-qz8h` | [NYC Open Data](https://data.cityofnewyork.us/Transportation/Serious-Injury-Response-Tracking-Analysis-Program-/xeqp-qz8h/about_data) |

Both datasets are downloaded in full each time the notebook runs (section 3), so the portal may have newer rows than the numbers in this notebook.

### 12.2 Shared definitions

| Term | Meaning |
|---|---|
| Serious harm | Total Severe Injuries + Total Fatalities, summed over crashes |
| Serious crash | One SIRTA record (SIRTA only records crashes with a severe injury or death) |
| Completed units | All Counted Units (affordable units), dated by Building Completion Date |
| Building size | Total Units (affordable + market-rate) |
| Analysis window | 1 Jan 2022 – 30 Jun 2026; full years = 2022–2025 |

### 12.3 AI prompt log

| # | Date | Member | Tool | Prompt (short) | Used for | What we kept / changed |
|---|---|---|---|---|---|---|
| 1 | 2026-11-04 | Baraa Tantawy | Claude (Claude Code) | "Read the handoff, project description and data catalog; tell me where we are and what I should do" | Project status check against the repo | Kept the status list and task order |
| 2 | 2026-11-04 | Baraa Tantawy | Claude (Claude Code) | "See what is better practice" for the severity filter | Decision D30 (severity switches the measure) | Kept; labels renamed to "Severe injuries" / "Deaths" |
| 3 | 2026-11-04 | Baraa Tantawy | Claude (Claude Code) | "Set up the venv and plotly"; pin versions for Vercel | Python 3.12 venv, pinned `requirements.txt` / `requirements-dev.txt` (D32); SSL fix with `pip-system-certs` | Kept |
| 4 | 2026-11-04 | Ali Issa | Claude (Claude Code) | SIRTA overview and EDA (section 5): missing values, empty rows, borough, dates/times, injury counts, lookback, ZIP/coordinates, duplicates, street names | Section 5 code, charts and 5.x summary | Kept; the 5.x handling column is a suggestion for section 7 |
| 5 | 2026-11-04 | Ali Issa | Claude (Claude Code) | Read the reviewer's change request on PR #3, say whether each point is right (checked against the brief), and apply the fixes | Section 5: explanation after every code cell, ZIP 10000 check and map, ST / highway wording, px import, regex pattern variable | Kept; all 8 review points accepted |
| 6 | 2026-11-04 | Mostafa Alaa | Claude (Claude Code) | Housing pre-integration cleaning (section 6) from the 4.x issue list: check each issue in code, decide drop / fill / keep with reasons and alternatives, standardise formats, export a clean buildings table with data-dictionary names | Section 6 code, markdown and 6.x decisions table; decisions D34-D41 | Kept; numbers checked against cell output |
